In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime

In [0]:
spark.conf.set(
    "fs.azure.account.key.adlsairbnbde.dfs.core.windows.net",
    "KEY HERE"
)

In [0]:
STORAGE_ACCOUNT = "adlsairbnbde"
BRONZE_BASE = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/airbnb"
 
CITIES = ["barcelona", "lisbon"]
QUARTERS_BY_CITY = {
    "barcelona": ["2025-Q3", "2026-Q2"],
    "lisbon": ["2025-Q3", "2026-Q2"],
}

In [0]:

expected_paths = []
 
for city in CITIES:
    for quarter in QUARTERS_BY_CITY[city]:
        expected_paths.append(f"{BRONZE_BASE}/listings/city={city}/quarter={quarter}/listings.csv.gz")
        expected_paths.append(f"{BRONZE_BASE}/calendar/city={city}/quarter={quarter}/calendar.csv.gz")
    expected_paths.append(f"{BRONZE_BASE}/neighbourhoods/city={city}/neighbourhoods.csv")
 
missing_files = []
found_files = []
 
for path in expected_paths:
    try:
        file_info = dbutils.fs.ls(path)
        found_files.append((path, file_info[0].size))
    except Exception:
        missing_files.append(path)
 
print(f"Found: {len(found_files)}/{len(expected_paths)} expected files")
 
if missing_files:
    print("\nMISSING FILES:")
    for f in missing_files:
        print(f"  - {f}")
    raise FileNotFoundError(
        f"{len(missing_files)} expected Bronze file(s) are missing. "
        f"Check that scripts/local_ingest_to_bronze.py completed successfully "
        f"for all cities/quarters before running this notebook."
    )
else:
    print("\nAll expected files present. Sizes:")
    for path, size in found_files:
        size_mb = size / (1024 * 1024)
        print(f"  {size_mb:8.2f} MB  {path}")
 

Found: 10/10 expected files

All expected files present. Sizes:
      9.26 MB  abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/listings/city=barcelona/quarter=2025-Q3/listings.csv.gz
     15.83 MB  abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/city=barcelona/quarter=2025-Q3/calendar.csv.gz
      7.80 MB  abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/listings/city=barcelona/quarter=2026-Q2/listings.csv.gz
     12.68 MB  abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/city=barcelona/quarter=2026-Q2/calendar.csv.gz
      0.00 MB  abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/neighbourhoods/city=barcelona/neighbourhoods.csv
     13.59 MB  abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/listings/city=lisbon/quarter=2025-Q3/listings.csv.gz
     20.71 MB  abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/city=lisbon/quarter=2025-Q3/calendar.csv.gz
     13.61 MB  abfss://bronze@adlsairbnbde.dfs.core.windows.

In [0]:
EXPECTED_LISTINGS_COLUMNS = {
    "id", "listing_url", "name", "host_id", "host_since",
    "host_is_superhost", "host_listings_count", "neighbourhood_cleansed",
    "room_type", "accommodates", "bedrooms", "bathrooms_text",
    "price", "number_of_reviews", "review_scores_rating",
}
 
EXPECTED_CALENDAR_COLUMNS = {
    "listing_id", "date", "available", "price", "adjusted_price", "minimum_nights",
}

In [0]:

def read_listings_raw(city, quarter):
    path = f"{BRONZE_BASE}/listings/city={city}/quarter={quarter}/listings.csv.gz"
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(path)
    )
    actual_columns = set(df.columns)
    missing_cols = EXPECTED_LISTINGS_COLUMNS - actual_columns
    if missing_cols:
        print(f"  WARNING [listings/{city}/{quarter}]: missing expected columns: {missing_cols}")
    df = df.withColumn("_source_city", F.lit(city)) \
           .withColumn("_source_quarter", F.lit(quarter)) \
           .withColumn("_ingested_at", F.lit(datetime.utcnow().isoformat()))
    return df
 
listings_dfs = []
for city in CITIES:
    for quarter in QUARTERS_BY_CITY[city]:
        print(f"Reading listings: {city} / {quarter}")
        listings_dfs.append(read_listings_raw(city, quarter))

Reading listings: barcelona / 2025-Q3
Reading listings: barcelona / 2026-Q2
Reading listings: lisbon / 2025-Q3
Reading listings: lisbon / 2026-Q2


In [0]:
bronze_listings_all = listings_dfs[0]
for df in listings_dfs[1:]:
    bronze_listings_all = bronze_listings_all.unionByName(df, allowMissingColumns=True)
 
print(f"\nTotal listings rows across all snapshots: {bronze_listings_all.count()}")
display(bronze_listings_all.groupBy("_source_city", "_source_quarter").count())


Total listings rows across all snapshots: 85028


_source_city,_source_quarter,count
barcelona,2025-Q3,19410
barcelona,2026-Q2,15293
lisbon,2025-Q3,25449
lisbon,2026-Q2,24876


In [0]:

def read_calendar_raw(city, quarter):
    path = f"{BRONZE_BASE}/calendar/city={city}/quarter={quarter}/calendar.csv.gz"
    df = (
        spark.read
        .option("header", True)
        .csv(path)
    )
    actual_columns = set(df.columns)
    missing_cols = EXPECTED_CALENDAR_COLUMNS - actual_columns
    if missing_cols:
        print(f"  WARNING [calendar/{city}/{quarter}]: missing expected columns: {missing_cols}")
    df = df.withColumn("_source_city", F.lit(city)) \
           .withColumn("_source_quarter", F.lit(quarter)) \
           .withColumn("_ingested_at", F.lit(datetime.utcnow().isoformat()))
    return df
 
calendar_dfs = []
for city in CITIES:
    for quarter in QUARTERS_BY_CITY[city]:
        print(f"Reading calendar: {city} / {quarter}")
        calendar_dfs.append(read_calendar_raw(city, quarter))
 
bronze_calendar_all = calendar_dfs[0]
for df in calendar_dfs[1:]:
    bronze_calendar_all = bronze_calendar_all.unionByName(df, allowMissingColumns=True)
 
print(f"\nTotal calendar rows across all snapshots: {bronze_calendar_all.count()}")
display(bronze_calendar_all.groupBy("_source_city", "_source_quarter").count())
 

Reading calendar: barcelona / 2025-Q3
Reading calendar: barcelona / 2026-Q2
  WARNING [calendar/barcelona/2026-Q2]: missing expected columns: {'adjusted_price', 'price'}
Reading calendar: lisbon / 2025-Q3
Reading calendar: lisbon / 2026-Q2
  WARNING [calendar/lisbon/2026-Q2]: missing expected columns: {'adjusted_price', 'price'}

Total calendar rows across all snapshots: 31089612


_source_city,_source_quarter,count
barcelona,2025-Q3,7084654
barcelona,2026-Q2,5623190
lisbon,2025-Q3,9288887
lisbon,2026-Q2,9092881


In [0]:

def read_neighbourhoods_raw(city):
    path = f"{BRONZE_BASE}/neighbourhoods/city={city}/neighbourhoods.csv"
    df = spark.read.option("header", True).csv(path)
    df = df.withColumn("_source_city", F.lit(city))
    return df
 
neighbourhood_dfs = [read_neighbourhoods_raw(city) for city in CITIES]
bronze_neighbourhoods_all = neighbourhood_dfs[0]
for df in neighbourhood_dfs[1:]:
    bronze_neighbourhoods_all = bronze_neighbourhoods_all.unionByName(df, allowMissingColumns=True)
 
print(f"Total neighbourhood rows: {bronze_neighbourhoods_all.count()}")
display(bronze_neighbourhoods_all.groupBy("_source_city").count())

Total neighbourhood rows: 207


_source_city,count
barcelona,73
lisbon,134


In [0]:

audit_rows = []
for path, size in found_files:
    audit_rows.append({
        "file_path": path,
        "size_bytes": size,
        "validated_at": datetime.utcnow().isoformat(),
        "status": "SUCCESS",
    })
 
audit_schema = StructType([
    StructField("file_path", StringType(), False),
    StructField("size_bytes", StringType(), False),  # kept as string for simplicity across mixed types
    StructField("validated_at", StringType(), False),
    StructField("status", StringType(), False),
])
 
audit_df = spark.createDataFrame(
    [(r["file_path"], str(r["size_bytes"]), r["validated_at"], r["status"]) for r in audit_rows],
    schema=audit_schema,
)
 
audit_path = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_bronze_audit"
audit_df.write.format("delta").mode("append").save(audit_path)
 
print(f"Audit log written to: {audit_path}")
display(audit_df)
 

Audit log written to: abfss://bronze@adlsairbnbde.dfs.core.windows.net/_bronze_audit


file_path,size_bytes,validated_at,status
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/listings/city=barcelona/quarter=2025-Q3/listings.csv.gz,9709188,2026-07-27T11:09:22.838179,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/city=barcelona/quarter=2025-Q3/calendar.csv.gz,16600259,2026-07-27T11:09:22.838192,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/listings/city=barcelona/quarter=2026-Q2/listings.csv.gz,8175620,2026-07-27T11:09:22.838199,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/city=barcelona/quarter=2026-Q2/calendar.csv.gz,13298828,2026-07-27T11:09:22.838205,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/neighbourhoods/city=barcelona/neighbourhoods.csv,2291,2026-07-27T11:09:22.838210,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/listings/city=lisbon/quarter=2025-Q3/listings.csv.gz,14248985,2026-07-27T11:09:22.838217,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/city=lisbon/quarter=2025-Q3/calendar.csv.gz,21718037,2026-07-27T11:09:22.838223,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/listings/city=lisbon/quarter=2026-Q2/listings.csv.gz,14273494,2026-07-27T11:09:22.838229,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/city=lisbon/quarter=2026-Q2/calendar.csv.gz,21813511,2026-07-27T11:09:22.838234,SUCCESS
abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/neighbourhoods/city=lisbon/neighbourhoods.csv,3720,2026-07-27T11:09:22.838240,SUCCESS


In [0]:
spark.read.option("header", True).csv(
    "abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/city=barcelona/quarter=2026-Q2/calendar.csv.gz"
).columns

['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']

In [0]:

bronze_listings_all.write.format("delta").mode("overwrite").save(
    f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_validated/listings"
)
bronze_calendar_all.write.format("delta").mode("overwrite").save(
    f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_validated/calendar"
)
bronze_neighbourhoods_all.write.format("delta").mode("overwrite").save(
    f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_validated/neighbourhoods"
)
print("Validated Bronze data persisted. Ready for 02_silver_clean_listings.py")
 

Validated Bronze data persisted. Ready for 02_silver_clean_listings.py
